# 7.4 · 交叉验证策略 / Cross-Validation Strategies

> **课程定位 / Where this fits**
> 第 4 课，**Part 7 · 模型评估与优化**。
> Lesson 4, **Part 7 · Model Evaluation & Tuning**.
>
> 3.10 已经详讲过各种 CV（从防泄漏角度）。这一课从**评估与选择**的角度**系统整合**：何时用哪种 CV（决策流程）、重复 CV/LOOCV、以及一个常被忽略的关键区分——**"用 CV 选模型"和"用 CV 估性能"是两回事**。CV 是评估的命脉，选错策略会让你对模型能力的判断整个失真。
> 3.10 covered CV in depth (from the leakage angle). This lesson **consolidates** it from the **evaluation & selection** angle: which CV when (a decision flow), repeated/LOOCV, and an often-missed distinction — **"CV for model selection" vs "CV for performance estimation" are different things**. CV is the lifeblood of evaluation; the wrong strategy distorts your whole judgment of the model.
>
> 💼 **实战/面试视角**："为什么交叉验证 / 各种 CV 怎么选 / 嵌套 CV 解决什么" 必考。
> 💼 **Practical/interview angle:** "why CV / choosing among CV variants / what nested CV solves" — must-knows.

> 💡 **面试相关 / Interview-relevant**
> - "为什么 K-fold 比单次划分好 / K 怎么选"（出镜率 ★★★★★）
> - "Stratified / Group / TimeSeries CV 何时用"（★★★★★）
> - "嵌套 CV 解决什么（调参乐观偏差）"（★★★★★）
> - "LOOCV 的优缺点"（★★★）
> - "CV 选模型 vs CV 估性能的区别"（★★★★）

---

## 学习目标 / Learning Objectives

1. 巩固"为什么 K-fold"+ K 的偏差方差权衡。
   Reinforce "why K-fold" and the bias-variance of K itself.
2. 用**决策流程**选对 CV 变体（分层/分组/时序）。
   Choose the right CV variant via a decision flow.
3. 了解 **RepeatedKFold / LOOCV** 的取舍。
   Know the trade-offs of RepeatedKFold / LOOCV.
4. 区分 **CV 选模型 vs CV 估性能**，用嵌套 CV 解决。
   Distinguish model selection vs performance estimation; solve with nested CV.

## 目录 / TOC
1. [先建直觉：CV 在估什么 ⭐](#1)
2. [K 的偏差方差 + LOOCV/Repeated ⭐](#2)
3. [选哪种 CV：决策流程 ⭐](#3)
4. [选模型 vs 估性能 + 嵌套 CV ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 先建直觉：CV 在估什么 ⭐ / Intuition: What CV Estimates

我们真正想知道的是模型在**未见数据**上的表现（泛化误差）。单次 train/test 划分只给一个数，且受"恰好分到哪些点"的运气影响。**K 折交叉验证**把数据切 K 份、轮流验证、取平均——让**每个样本都当过一次验证**，得到更稳的估计 + 一个标准差（告诉你这个估计本身有多可靠）。
What we actually want is performance on **unseen data** (generalization error). A single train/test split gives one number, swayed by the luck of which points landed where. **K-fold CV** splits into K parts, rotates the validation fold, and averages — so **every sample validates once**, giving a steadier estimate plus a standard deviation (telling you how reliable that estimate is).

关键原则（贯穿全课）：**任何"会看数据学东西"的步骤（缩放、填补、选特征、调参）都必须在每折的训练部分内做**，否则泄漏（3.9/3.12）。CV 不只是"算个平均分"，它是在**模拟真实部署**。
Key principle (throughout): **any step that "learns from data" (scaling, imputing, feature selection, tuning) must happen inside each fold's training part**, or you leak (3.9/3.12). CV isn't just "averaging scores" — it **simulates real deployment**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
sns.set_theme(style="whitegrid")

X, y = load_iris(return_X_y=True)

# 单次划分的运气波动 vs K-fold 的稳定 / single-split luck vs K-fold stability
single = [train_test_split(X, y, test_size=0.3, random_state=s) for s in range(30)]
single_acc = [LogisticRegression(max_iter=500).fit(a, c).score(b, d) for a,b,c,d in single]
cv = cross_val_score(LogisticRegression(max_iter=500), X, y, cv=5)
print(f"30 次单次划分的 test 准确率: 范围 [{min(single_acc):.2f}, {max(single_acc):.2f}], std={np.std(single_acc):.3f}")
print(f"5 折 CV: {cv.mean():.3f} ± {cv.std():.3f}  (一次给出均值+可靠性, 每样本都验证过)")


<a id="2"></a>
## 2. K 的偏差方差 + LOOCV/Repeated ⭐ / K's Bias-Variance & Variants

**K 本身也有偏差方差权衡**（容易被忽略的考点）：
**K itself has a bias-variance trade-off** (an easily-missed point):
- **K 小（如 3）**：每折训练数据少 → 估计偏**悲观**（偏差大），但折间差异小、计算快。
  **Small K (e.g. 3):** less training data per fold → pessimistic estimate (more bias), but folds are similar and it's fast.
- **K 大（如 LOOCV，K=n）**：每折几乎用全部数据训练 → 估计**近无偏**，但 n 个模型的训练集高度重叠 → 估计**方差大**、且要训 n 次极慢。
  **Large K (LOOCV, K=n):** nearly all data per fold → near-unbiased, but the n training sets overlap heavily → high variance, and training n times is very slow.
- **K=5 或 10** 是公认的好折中。
  **K=5 or 10** is the accepted sweet spot.
- **RepeatedKFold**：用不同随机划分重复多次 K 折取平均，进一步降低"恰好这次怎么分"的随机性——数据不大时很值。
  **RepeatedKFold:** repeats K-fold with different random splits and averages, further reducing split randomness — worthwhile on smaller data.


In [ ]:
from sklearn.model_selection import KFold, LeaveOneOut, RepeatedKFold
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
import time

Xd, yd = load_diabetes(return_X_y=True)        # 一个小回归数据集 / small regression dataset
model = Ridge(alpha=1.0)

print(f"{'策略 strategy':<22} {'估计 R²':>9} {'std':>7} {'耗时s':>7}")
for name, cvobj in [("3-fold", KFold(3, shuffle=True, random_state=0)),
                    ("5-fold", KFold(5, shuffle=True, random_state=0)),
                    ("10-fold", KFold(10, shuffle=True, random_state=0)),
                    ("Repeated 5x10", RepeatedKFold(n_splits=10, n_repeats=5, random_state=0)),
                    ("LOOCV (K=n)", LeaveOneOut())]:
    t = time.perf_counter()
    s = cross_val_score(model, Xd, yd, cv=cvobj, scoring="r2")
    print(f"{name:<22} {s.mean():>9.3f} {s.std():>7.3f} {time.perf_counter()-t:>7.3f}")
print("\nK 小→偏悲观; K=5/10 是好折中; Repeated 更稳(降随机); LOOCV 近无偏但慢且方差大")


<a id="3"></a>
## 3. 选哪种 CV：决策流程 ⭐ / Which CV: A Decision Flow

普通 K-fold 假设样本**独立同分布且可随机打乱**。一旦这个假设破了，就要换专门的 CV（详见 3.10），否则会**泄漏 + 估计虚高**。决策流程：
Plain K-fold assumes samples are **i.i.d. and shufflable**. When that breaks, switch to a specialized CV (detailed in 3.10), or you'll **leak and overestimate**. The decision flow:

```
数据有时间顺序(时序)?  ──是──> TimeSeriesSplit (train 永在 val 之前)
        │否
有重复实体(用户/病人/设备)? ──是──> GroupKFold (同实体锁定一折)
        │否
是分类且类别不平衡?     ──是──> StratifiedKFold (保持各折类别比例)
        │否
                       ──> 普通 KFold(shuffle=True)
```

下面用代码各跑一个，确认它们的"保护"确实生效。
Below we run each and confirm its "protection" actually holds.


In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold, TimeSeriesSplit
rng = np.random.default_rng(0)

# 分层: 不平衡数据各折保持比例 / stratified keeps class ratio
y_imb = np.r_[np.zeros(180), np.ones(20)]; X_imb = rng.normal(size=(200, 3))
ratios = [y_imb[te].mean() for _, te in StratifiedKFold(5).split(X_imb, y_imb)]
print(f"StratifiedKFold 各折正类比例: {[f'{r:.2f}' for r in ratios]} (整体 {y_imb.mean():.2f}, 严格保持)")

# 分组: 同一实体不跨 train/test / group keeps an entity on one side
groups = np.repeat(np.arange(40), 5)
shared = [len(set(groups[tr]) & set(groups[te])) for tr, te in GroupKFold(5).split(X_imb[:200], groups=groups)]
print(f"GroupKFold 每折 train/test 共享的实体数: {shared} (全 0 → 无泄漏)")

# 时序: train 索引永远小于 val / time-series keeps train before val
ok = all(tr.max() < te.min() for tr, te in TimeSeriesSplit(5).split(np.arange(100)))
print(f"TimeSeriesSplit: 每折 train 都在 val 之前? {ok} (用过去预测未来)")


<a id="4"></a>
## 4. 选模型 vs 估性能 + 嵌套 CV ⭐ / Selection vs Estimation & Nested CV

一个被严重低估的区分（高阶面试点）：
A badly underrated distinction (advanced interview point):
- **用 CV 选模型/调参**：从一堆候选里挑 CV 分数最高的——这没问题。
  **CV for model selection/tuning:** pick the candidate with the highest CV score — fine.
- **用同一个 CV 报告"被选中模型"的性能**：**有问题**！因为你是专门挑了"在这份 CV 上运气最好的那个"，这个 CV 分数**偏乐观**。
  **Reporting that selected model's performance with the same CV:** **a problem!** You picked "whatever got luckiest on this CV", so that score is **optimistic**.

**嵌套 CV** 解决它：**内层 CV 调参，外层 CV 评估**——外层每一折都把"调参"当作模型的一部分重做一遍，于是评估时从没见过被选中的超参，得到调参后的**无偏**估计。这是论文/汇报模型真实能力的金标准。
**Nested CV** fixes it: **inner CV tunes, outer CV evaluates** — each outer fold redoes tuning as part of the model, so evaluation never saw the chosen hyperparameters, giving an **unbiased** post-tuning estimate. The gold standard for reporting true ability.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.datasets import make_classification

# 有噪声的小数据, 让乐观偏差显现 / noisy small data so the bias shows
Xn, yn = make_classification(n_samples=200, n_features=20, n_informative=5, flip_y=0.15, random_state=0)
grid = {"C": np.logspace(-2, 2, 10)}

non_nested, nested = [], []
for seed in range(8):                              # 多种子平均, 让差异稳定 / average over seeds
    inner = StratifiedKFold(5, shuffle=True, random_state=seed)
    outer = StratifiedKFold(5, shuffle=True, random_state=seed)
    gs = GridSearchCV(SVC(), grid, cv=inner).fit(Xn, yn)
    non_nested.append(gs.best_score_)              # 用同一CV调参+报分 → 乐观
    nested.append(cross_val_score(gs, Xn, yn, cv=outer).mean())  # 外层评估含调参的整个流程
print(f"非嵌套 CV (调参+报分同一CV): {np.mean(non_nested):.3f}  ← 乐观偏差")
print(f"嵌套 CV   (内层调参+外层评估): {np.mean(nested):.3f}  ← 无偏估计")
print(f"差异 {(np.mean(non_nested)-np.mean(nested))*100:.1f} 个百分点 = 调参偷看带来的乐观偏差")
print("→ 汇报真实能力用嵌套CV; 选完超参再用全部训练数据重训上线")


<a id="5"></a>
## 5. 小结 / Summary

```
CV 目的: 估泛化误差; K-fold 让每样本都验证一次, 给均值±std(比单次划分稳)
K 的权衡: 小K 偏悲观但快; LOOCV 近无偏但慢+方差大; K=5/10 折中; RepeatedKFold 更稳
选 CV(决策流): 时序→TimeSeriesSplit; 重复实体→GroupKFold; 不平衡分类→StratifiedKFold; 否则 KFold(shuffle)
预处理/调参必须在每折训练部分内做(Pipeline 保证, 3.12) → 防泄漏
选模型 vs 估性能: 同一CV既调参又报分 → 乐观偏差; 嵌套CV(内调参+外评估)给无偏估计
```

### 💡 面试速查 / Interview cheat-sheet
1. **K-fold 比单次划分稳**; K=5/10 是偏差方差折中; LOOCV 近无偏但慢+高方差。
   K-fold beats a single split; K=5/10 balances bias-variance; LOOCV is near-unbiased but slow/high-variance.
2. **时序→TimeSeriesSplit, 重复实体→GroupKFold, 不平衡→StratifiedKFold**。
   Time-series → TimeSeriesSplit, repeated entities → GroupKFold, imbalance → StratifiedKFold.
3. **预处理/调参只在每折训练部分**(Pipeline), 否则泄漏。
   Preprocessing/tuning only in each fold's train part (Pipeline), else leakage.
4. **嵌套 CV** 给调参后的无偏估计(同一CV调参+报分会乐观)。
   Nested CV gives an unbiased post-tuning estimate (same-CV tuning+reporting is optimistic).
5. **CV 选模型 ≠ CV 估性能**, 别混用同一份 CV。
   CV for selection ≠ CV for estimation; don't reuse the same CV for both.

### 下一节 / Next
**7.5 超参数调优**——有了可靠的 CV, 就能系统地搜超参: 网格搜索、随机搜索、贝叶斯优化(Optuna), 以及它们的效率对比。
**7.5 Hyperparameter Tuning** — with reliable CV, search hyperparameters systematically: grid, random, and Bayesian optimization (Optuna), with efficiency comparisons.
